In [110]:
# vars 
_1src = "/Users/yerik/Music/_1_NEW_SOURCE"

In [111]:
from pathlib import Path

folder = Path(_1src)

# ONLY count valid AIFF (strict check)
aiff_ext = {".aiff", ".aif"}

files = [f for f in folder.rglob("*") if f.is_file()]

count_total = len(files)

count_aiff = len([
    f for f in files
    if f.suffix.lower() in aiff_ext
    and not f.name.startswith("._")   # avoid mac artifacts
])

count_other = count_total - count_aiff

print(f"TOTAL FILES : {count_total}")
print(f"AIFF FILES  : {count_aiff}")
print(f"OTHER FILES : {count_other}")

TOTAL FILES : 2446
AIFF FILES  : 2402
OTHER FILES : 44


# How mamny of those have id ?

In [98]:
# count AIFF files WITH valid ID pattern + build df_ids

import re
from pathlib import Path
import pandas as pd

folder = Path(_1src)

pattern = re.compile(r"--id_(.*?)(?:--|$)")

aiff_files = [
    f for f in folder.rglob("*")
    if f.is_file()
    and f.suffix.lower() in [".aiff", ".aif"]
    and not f.name.startswith("._")
]

count_total_aiff = len(aiff_files)

# ---------------------------------
# BUILD DF WITH ID + PATH
# ---------------------------------
rows = []

for f in aiff_files:
    match = pattern.search(f.name)
    if match:
        id_clean = match.group(1).strip("-")
        rows.append({
            "ID": id_clean,
            "Path": str(f)
        })

df_ids = pd.DataFrame(rows)

count_with_id = len(df_ids)
count_without_id = count_total_aiff - count_with_id

# ---------------------------------
# PRINT
# ---------------------------------
print(f"AIFF TOTAL     : {count_total_aiff}")
print(f"WITH ID        : {count_with_id}")
print(f"WITHOUT ID     : {count_without_id}")

print(len(df_ids))

AIFF TOTAL     : 2406
WITH ID        : 2406
WITHOUT ID     : 0
2406


In [99]:
# ---------------------------------
# CLEAN IDS (BOTH DFS)
# ---------------------------------
df_ids['ID_clean'] = (
    df_ids['ID']
    .astype(str)
    .str.strip()
    .str.lower()
)

df['ID_clean'] = (
    df['ID']
    .astype(str)
    .str.strip()
    .str.lower()
)

# ---------------------------------
# MERGE → FIND MISMATCHES
# ---------------------------------
df_merge = df_ids.merge(
    df[['ID_clean']],
    on='ID_clean',
    how='outer',
    indicator=True
)

# ---------------------------------
# SPLIT RESULTS
# ---------------------------------
df_ids_unmatched = df_merge[df_merge['_merge'] == 'left_only'].copy()
df_unmatched     = df_merge[df_merge['_merge'] == 'right_only'].copy()

# ---------------------------------
# PRINT
# ---------------------------------
print(f"df_ids unmatched : {len(df_ids_unmatched)}")
print(f"df unmatched     : {len(df_unmatched)}")

# optional view
pd.set_option('display.max_colwidth', None)
df_ids_unmatched.head()
df_unmatched.head()

df_ids unmatched : 4
df unmatched     : 8


,ID,Path,ID_clean,_merge
49,NaN,NaN,t1-103103,right_only
122,NaN,NaN,t1-11140n,right_only
194,NaN,NaN,t1-121901,right_only
198,NaN,NaN,t1-121905,right_only
209,NaN,NaN,t1-12190g,right_only


In [105]:
check_df = df_unmatched[['ID','Path']]
check_df 

,ID,Path
49,NaN,NaN
122,NaN,NaN
194,NaN,NaN
198,NaN,NaN
209,NaN,NaN
901,NaN,NaN
966,NaN,NaN
1797,NaN,NaN


In [107]:
df_ids_unmatched[['ID','Path']]

,ID,Path
159,t1-11600,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_GreenL_Puma/dylu_3U[25]-120BPM-6B_A#maj--id_t1-11600---Deep-METROPOLI--by--OMICHMATTEI-saltyoriginalm(O)-2013.aiff
160,t1-11601,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_GreenL_Puma/dylu_3U[25]-120BPM-8A_Amin--id_t1-11601---Deep-BACCARAM--by--OMICHMATTEI-loopertwoorigi(O)-2014.aiff
161,t1-11603,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_GreenL_Puma/dylu_3S[25]-140BPM-2A_D#min--id_t1-11603---Bass-TRATRATRA--by--NICKLEONELAMI-ghostorchidori(O)-2025.aiff
162,t1-11604,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_GreenL_Puma/dylu_3W[25]-98BPM-4B_G#maj--id_t1-11604---Lati-WEALATIN--by--MATTEIARCANGEL-palasgirlasco(U)-2025.aiff


In [100]:
df[df['ID'] == 't1-11140n']

,Path,temp_id,file_name,Extension,dur_seconds,dur_min,sr,bit_depth,bit_rate,channels,...,Path_csv_freq,Path_png_dr,Path_png_id_and_key,year_written_id3,bought_year,lufs_pct,re_name,audio_hash,__source_file,ID_clean
102,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_11_WELLd_Spkr_PUMA_most/TRkw_Take_on_Me_ARkw_A-Ha_MXkw_2016_Remaster_KYkw_A_Major_BPkw_85_GNkw_Pop_RMkw__LBkw_Rhino_Warner_Records_RYkw_2020_02_26_PYkw_2025_10_31.aiff,TRkw_Take_on_Me_ARkw_A-Ha_MXkw_2016_Remaster_KYkw_A_Major_BPkw_85_GNkw_Pop_RMkw__LBkw_Rhino_Warner_Records_RYkw_2020_02_26_PYkw_2025_10_31,TRkw_Take_on_Me_ARkw_A-Ha_MXkw_2016_Remaster_KYkw_A_Major_BPkw_85_GNkw_Pop_RMkw__LBkw_Rhino_Warner_Records_RYkw_2020_02_26_PYkw_2025_10_31.aiff,.aiff,228.295193,3.80492,44100,16,None,2,...,tables/table_freq_t1-11140n.csv,images/dbs_plot_t1-11140n.png,images/key_and_id_t1-11140n.png,Unsupported,2025,91,dylu_6U[25]-85BPM-11B_Amaj--id_t1-11140n---Pop-RHINO/WAR--by--AHA-takeonme2016(U)-2020,a48a48e5c058de23a1c640ee3f2c943bb9fdecad5dc8ac63657119047429ef05,df_WDGt_final.pkl,t1-11140n


In [85]:
len(df_ids['ID'])

2406

In [86]:
# how many IDs match between df and df_ids
df_ids['ID'].isin(df['ID']).sum()

2402

In [87]:
pd.set_option('display.max_colwidth', None)

# mismatches (IDs in df_ids NOT in df) with full path
df_ids[~df_ids['ID'].isin(df['ID'])][['ID', 'Path']]

,ID,Path
1267,t1-11600,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_GreenL_Puma/dylu_3U[25]-120BPM-6B_A#maj--id_t1-11600---Deep-METROPOLI--by--OMICHMATTEI-saltyoriginalm(O)-2013.aiff
1272,t1-11601,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_GreenL_Puma/dylu_3U[25]-120BPM-8A_Amin--id_t1-11601---Deep-BACCARAM--by--OMICHMATTEI-loopertwoorigi(O)-2014.aiff
1287,t1-11603,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_GreenL_Puma/dylu_3S[25]-140BPM-2A_D#min--id_t1-11603---Bass-TRATRATRA--by--NICKLEONELAMI-ghostorchidori(O)-2025.aiff
1291,t1-11604,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_GreenL_Puma/dylu_3W[25]-98BPM-4B_G#maj--id_t1-11604---Lati-WEALATIN--by--MATTEIARCANGEL-palasgirlasco(U)-2025.aiff


In [92]:
# ---------------------------------
# UNMATCHED FROM BOTH SIDES
# ---------------------------------

# normalize (just in case)
df_ids['ID'] = df_ids['ID'].astype(str).str.strip()
df['ID']     = df['ID'].astype(str).str.strip()

# ---------------------------------
# df_ids → NOT in df
# ---------------------------------
df_ids_unmatched = df_ids[~df_ids['ID'].isin(df['ID'])].copy()

# ---------------------------------
# df → NOT in df_ids
# ---------------------------------
df_unmatched = df[~df['ID'].isin(df_ids['ID'])].copy()

# ---------------------------------
# QUICK COUNTS
# ---------------------------------
print(f"df_ids unmatched : {len(df_ids_unmatched)}")
print(f"df unmatched     : {len(df_unmatched)}")

df_ids_unmatched.head()
df_unmatched.head()

df_ids unmatched : 4
df unmatched     : 8


,Path,temp_id,file_name,Extension,dur_seconds,dur_min,sr,bit_depth,bit_rate,channels,...,Path_png_bar_centroid,Path_csv_freq,Path_png_dr,Path_png_id_and_key,year_written_id3,bought_year,lufs_pct,re_name,audio_hash,__source_file
102,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_11_WELLd_Spkr_PUMA_most/TRkw_Take_on_Me_ARkw_A-Ha_MXkw_2016_Remaster_KYkw_A_Major_BPkw_85_GNkw_Pop_RMkw__LBkw_Rhino_Warner_Records_RYkw_2020_02_26_PYkw_2025_10_31.aiff,TRkw_Take_on_Me_ARkw_A-Ha_MXkw_2016_Remaster_KYkw_A_Major_BPkw_85_GNkw_Pop_RMkw__LBkw_Rhino_Warner_Records_RYkw_2020_02_26_PYkw_2025_10_31,TRkw_Take_on_Me_ARkw_A-Ha_MXkw_2016_Remaster_KYkw_A_Major_BPkw_85_GNkw_Pop_RMkw__LBkw_Rhino_Warner_Records_RYkw_2020_02_26_PYkw_2025_10_31.aiff,.aiff,228.295193,3.80492,44100,16,None,2,...,images/centroid_donut_t1-11140n.png,tables/table_freq_t1-11140n.csv,images/dbs_plot_t1-11140n.png,images/key_and_id_t1-11140n.png,Unsupported,2025,91,dylu_6U[25]-85BPM-11B_Amaj--id_t1-11140n---Pop-RHINO/WAR--by--AHA-takeonme2016(U)-2020,a48a48e5c058de23a1c640ee3f2c943bb9fdecad5dc8ac63657119047429ef05,df_WDGt_final.pkl
444,"/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_Minimal_jazzy_Housy/TRkw_Marnix_ARkw_Retromigration,_Nephews_MXkw_Original_Mix_KYkw_C_Minor_BPkw_120_GNkw_Deep_House_RMkw__LBkw_Ltd,_W_LBL_RYkw_2020_11_06_PYkw_2025_12_19.aiff","TRkw_Marnix_ARkw_Retromigration,_Nephews_MXkw_Original_Mix_KYkw_C_Minor_BPkw_120_GNkw_Deep_House_RMkw__LBkw_Ltd,_W_LBL_RYkw_2020_11_06_PYkw_2025_12_19","TRkw_Marnix_ARkw_Retromigration,_Nephews_MXkw_Original_Mix_KYkw_C_Minor_BPkw_120_GNkw_Deep_House_RMkw__LBkw_Ltd,_W_LBL_RYkw_2020_11_06_PYkw_2025_12_19.aiff",.aiff,431.839252,7.197321,44100,16,None,2,...,images/centroid_donut_t1-121901.png,tables/table_freq_t1-121901.csv,images/dbs_plot_t1-121901.png,images/key_and_id_t1-121901.png,Unsupported,2025,87,"dylu_0T[25]-120BPM-5A_Cmin--id_t1-121901---Deep-LTD,W/LB--by--RETROMIGRATIONN-marnixoriginal(O)-2020",85f2a92f4468acdb95ce42a47b44d507daed127e244f3b8fcb975864479b136c,df_mjht_final.pkl
448,"/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_Minimal_jazzy_Housy/TRkw_Kindness_ARkw_Retromigration,_Nephews_MXkw_Original_Mix_KYkw_A#_Minor_BPkw_121_GNkw_Deep_House_RMkw__LBkw_Ltd,_W_LBL_RYkw_2020_11_06_PYkw_2025_12_19.aiff","TRkw_Kindness_ARkw_Retromigration,_Nephews_MXkw_Original_Mix_KYkw_A#_Minor_BPkw_121_GNkw_Deep_House_RMkw__LBkw_Ltd,_W_LBL_RYkw_2020_11_06_PYkw_2025_12_19","TRkw_Kindness_ARkw_Retromigration,_Nephews_MXkw_Original_Mix_KYkw_A#_Minor_BPkw_121_GNkw_Deep_House_RMkw__LBkw_Ltd,_W_LBL_RYkw_2020_11_06_PYkw_2025_12_19.aiff",.aiff,396.501451,6.608358,44100,16,None,2,...,images/centroid_donut_t1-121905.png,tables/table_freq_t1-121905.csv,images/dbs_plot_t1-121905.png,images/key_and_id_t1-121905.png,Unsupported,2025,82,"dylu_0S[25]-120BPM-4B_G#maj--id_t1-121905---Deep-LTD,W/LB--by--RETROMIGRATIONN-kindnessorigina(O)-2020",6af91497317f1ed57018e7a931a0f85f2d007df0d675ae15c381f0c18fe3b597,df_mjht_final.pkl
459,"/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_Minimal_jazzy_Housy/TRkw_Never_Alone_ARkw_Retromigration,_Nephews_MXkw_Original_Mix_KYkw_A_Major_BPkw_121_GNkw_Deep_House_RMkw__LBkw_Ltd,_W_LBL_RYkw_2020_11_06_PYkw_2025_12_19.aiff","TRkw_Never_Alone_ARkw_Retromigration,_Nephews_MXkw_Original_Mix_KYkw_A_Major_BPkw_121_GNkw_Deep_House_RMkw__LBkw_Ltd,_W_LBL_RYkw_2020_11_06_PYkw_2025_12_19","TRkw_Never_Alone_ARkw_Retromigration,_Nephews_MXkw_Original_Mix_KYkw_A_Major_BPkw_121_GNkw_Deep_House_RMkw__LBkw_Ltd,_W_LBL_RYkw_2020_11_06_PYkw_2025_12_19.aiff",.aiff,366.048254,6.100804,44100,16,None,2,...,images/centroid_donut_t1-12190g.png,tables/table_freq_t1-12190g.csv,images/dbs_plot_t1-12190g.png,images/key_and_id_t1-12190g.png,Unsupported,2025,87,"dylu_0T[25]-120BPM-10A_Bmin--id_t1-12190g---Deep-LTD,W/LB--by--RETROMIGRATIONN-neveraloneorig(O)-2020",af83ad4de237975bde98951d309866f11ca119111bcaa95377ff0c51bf1f307a,df_mjht_final.pkl
593,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_08_BF25-EcsaticDance/TRkw_Sex_on_Fire_ARkw

In [41]:
df['ID']

0       t1-122700
1       t1-122701
2       t1-122702
3       t2-122700
4       t2-122701
          ...    
2405      t1-660k
2406      t1-660l
2407      t1-660m
2408      t1-660n
2409      t1-660o
Name: ID, Length: 2410, dtype: object

# matching 

In [31]:
def EDA_build_id_df(folder):
    """
    Extract ID from '--id_' until next '--'
    """

    import os
    import re
    import pandas as pd

    rows = []

    # flexible: stop at next '--'
    pattern = re.compile(r"--id_(.*?)(?:--|$)")

    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".aiff") and not f.startswith("._"):

                match = pattern.search(f)

                if match:
                    id_clean = match.group(1).strip("-")  # clean extra '-'
                    full_path = os.path.join(root, f)

                    rows.append({
                        "ID": id_clean,
                        "Path": full_path
                    })

    df_files = pd.DataFrame(rows)

    print(f"\nTotal files: {len(df_files)}")

    return df_files

In [32]:
folder = "/Users/yerik/Music/_1_NEW_SOURCE"

df_files = EDA_build_id_df(folder)


Total files: 2406


In [33]:
#df_files['ID']

# check ids in dfs

In [43]:

# ---------------------------------------------------
# PKL CONCAT DATA LOADER MODULE
# ---------------------------------------------------

import pandas as pd
from pathlib import Path
from tqdm import tqdm


def load_pkl_folder(path):
    """
    Load and concatenate all PKL files from a folder (including subfolders).

    Parameters
    ----------
    path : str
        Root directory containing PKL files

    Returns
    -------
    df : pandas.DataFrame
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Folder not found: {path}")

    # ---------------------------------------------------
    # FIND PKL FILES
    # ---------------------------------------------------

    pkl_files = [p for p in path.rglob("*.pkl") if not p.name.startswith("._")]

    if len(pkl_files) == 0:
        raise ValueError("No PKL files found in directory")

    print(f"\nFound {len(pkl_files)} PKL files")

    # ---------------------------------------------------
    # LOAD + CONCAT
    # ---------------------------------------------------

    dfs = []

    for pkl in tqdm(pkl_files, desc="Loading PKLs"):
        try:
            df = pd.read_pickle(pkl)
            df["__source_file"] = pkl.name
            dfs.append(df)
        except Exception as e:
            print(f"Error loading {pkl}: {e}")

    if len(dfs) == 0:
        raise ValueError("No valid PKL files could be loaded")

    df = pd.concat(dfs, ignore_index=True)

    # ---------------------------------------------------
    # INFO
    # ---------------------------------------------------

    print("\nDataset Combined")
    print("---------------------------")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory usage: {mem:.2f} MB")

    return df


In [44]:
path = "/Users/yerik/_apple_lib/_a_progs/_a2ms_env/_9_ML_project/data/raw/_aiff_tracks_data"

df = load_pkl_folder(path)


Found 39 PKL files


Loading PKLs: 100%|██████████████████████████████████████████████████████████| 39/39 [00:00<00:00, 695.61it/s]


Dataset Combined
---------------------------
Rows: 2410
Columns: 76
Memory usage: 11.00 MB


In [82]:
df['ID'].duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
2405    False
2406    False
2407    False
2408    False
2409    False
Name: ID, Length: 2410, dtype: bool

In [13]:
matches = set(df['ID'].astype(str)).intersection(set(df_files['ID'].astype(str)))

print(f"Matches: {len(matches)}")

Matches: 2402


In [16]:
#matches

In [109]:
import os
import pandas as pd
from tqdm import tqdm

# ---------- INPUT ----------
root_folder = _1src
keyword_exclude = "dylu_"
audio_extensions = [".aiff", ".aif"]

# ---------- SCAN ----------
paths = []

for root, _, files in os.walk(root_folder):
    for file in tqdm(files, desc=f"Scanning: {root}"):
        
        # skip system junk
        if file.startswith("._") or file.startswith(".DS"):
            continue
        
        file_lower = file.lower()
        
        # check extension
        if any(file_lower.endswith(ext) for ext in audio_extensions):
            
            # EXCLUDE dylu_
            if not file_lower.startswith(keyword_exclude):
                full_path = os.path.join(root, file)
                paths.append(full_path)

# ---------- DF ----------
df = pd.DataFrame({"Path": paths})

# ---------- EXPORT ----------
output_path = os.path.join(root_folder, "_NO_dylu_files.csv")
df.to_csv(output_path, index=False)

print(f"\nTOTAL FILES (no dylu_): {len(df)}")
print(f"Saved → {output_path}")

Scanning: /Users/yerik/Music/_1_NEW_SOURCE: 100%|████████████████████████████| 2/2 [00:00<00:00, 20020.54it/s]
Scanning: /Users/yerik/Music/_1_NEW_SOURCE/_2026_this: 100%|█████████████████| 1/1 [00:00<00:00, 17476.27it/s]
Scanning: /Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Mini_VOCAL: 100%|█| 7/7 [00:00<00:00, 96898.11it
Scanning: /Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_deluxe_VOCAL: 100%|█| 10/10 [00:00<00:00, 203606
Scanning: /Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Anormal_house_mini: 100%|█| 10/10 [00:00<00:00, 
Scanning: /Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_03_Ashton_Joseph_SPKRlast: 100%|█| 174/174 [00:00<0
Scanning: /Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_NEW_Jackin: 100%|█| 16/16 [00:00<00:00, 291777.6
Scanning: /Users/yerik/Music/_1_NEW_SOURCE/_2023_this: 100%|██████████████████| 1/1 [00:00<00:00, 8848.74it/s]
Scanning: /Users/yerik/Music/_1_NEW_SOURCE/_2023_this/_23_09_AN_ArtPArk: 100%|█| 21/21 [00:00<00:00, 312341.79
S


TOTAL FILES (no dylu_): 0
Saved → /Users/yerik/Music/_1_NEW_SOURCE/_NO_dylu_files.csv


In [ ]:
def EDA_build_id_df(folder):
    """
    Build df with ID + Path from AIFF files
    Extract ID strictly between '--id_' and '----'
    """

    import os
    import re
    import pandas as pd

    rows = []

    pattern = re.compile(r"--id_(.*?)----")  # exact pattern

    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".aiff") and not f.startswith("._"):

                match = pattern.search(f)

                if match:
                    id_clean = match.group(1)  # e.g. t1-363o
                    full_path = os.path.join(root, f)

                    rows.append({
                        "ID": id_clean,
                        "Path": full_path
                    })

    df_files = pd.DataFrame(rows)

    print(f"\nTotal files: {len(df_files)}")

    return df_files

In [1]:
import pandas as pd
import os 
df_rk_aiff_25_07 = pd.read_pickle("_df_aiff_rk_1635_25_07.pkl")
df_exports_25_7 = pd.read_pickle("_df_exports_766_25_07.pkl")
df_STEMS_25_7 = pd.read_pickle("_df_STEMS_3286_25_07.pkl")
df_samples_STEMS_25_7 = pd.read_pickle("_df_samples_STEMS_72263_25_07.pkl")


In [2]:
print(len(df_rk_aiff_25_07),len(df_exports_25_7) , len(df_samples_STEMS_25_7),len(df_STEMS_25_7) )


1635 766 72263 3286


In [3]:
for name, df in [
    ('df_rk_aiff_25_07', df_rk_aiff_25_07),
    ('df_exports_25_7', df_exports_25_7),
    ('df_STEMS_25_7', df_STEMS_25_7),
    ('df_samples_STEMS_25_7', df_samples_STEMS_25_7)
]:
    
    total = len(df)
    missing = df['Path'].apply(lambda x: not os.path.exists(x) if pd.notna(x) else True).sum()
    print(f"🧪 {name:<25} → ❌ Missing: {missing:>6} / {total:<6} ✅ Exists: {total - missing}")

🧪 df_rk_aiff_25_07          → ❌ Missing:      0 / 1635   ✅ Exists: 1635
🧪 df_exports_25_7           → ❌ Missing:      0 / 766    ✅ Exists: 766
🧪 df_STEMS_25_7             → ❌ Missing:      0 / 3286   ✅ Exists: 3286
🧪 df_samples_STEMS_25_7     → ❌ Missing:      0 / 72263  ✅ Exists: 72263


# check paths 

In [4]:

#df_exports_25_7['Path'].head(5).tolist()
#/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATERIAL/_ALL_6_PROD_exports/_25-56_EXPORTS__

In [5]:
import pandas as pd

df_rk_aiff_25_07 = pd.read_pickle("_df_aiff_rk_1635_25_07.pkl")
df_exports_25_7 = pd.read_pickle("_df_exports_766_25_07.pkl")
df_STEMS_25_7 = pd.read_pickle("_df_STEMS_3286_25_07.pkl")
df_samples_STEMS_25_7 = pd.read_pickle("_df_samples_STEMS_72263_25_07.pkl")

for name, df, id_col in [
    ('df_rk_aiff_25_07', df_rk_aiff_25_07, 'ID'),
    ('df_exports_25_7', df_exports_25_7, 'ID'),
    ('df_STEMS_25_7', df_STEMS_25_7, 'ID'),
    ('df_samples_STEMS_25_7', df_samples_STEMS_25_7, 'id_stem_sample')
]:
    ids = df[id_col].head().to_list()
    print(f"{name:<25} → {ids}")


df_rk_aiff_25_07          → ['t1-61300', 't1-61301', 't1-61302', 't1-61303', 't1-61304']
df_exports_25_7           → ['e5600', 'e5602', 'e5603', 'e5604', 'e5605']
df_STEMS_25_7             → ['v1-71100', 'v1-71101', 'v1-71102', 'v1-71103', 'v1-71104']
df_samples_STEMS_25_7     → ['s15_b4-71429', 's24_b4-71429', 's22_b4-71429', 's11_b4-71429', 's83_b4-71429']


In [6]:
# -----######-----###### TOTAL UNIQUE IDs ACROSS ALL SETS -----######-----###### #
id_sets = set()

# Collect IDs from each DataFrame
id_sets.update(df_rk_aiff_25_07['ID'].dropna().unique())
id_sets.update(df_exports_25_7['ID'].dropna().unique())
id_sets.update(df_STEMS_25_7['ID'].dropna().unique())
id_sets.update(df_samples_STEMS_25_7['id_stem_sample'].dropna().unique())

print(f"🎯 Total unique IDs across all 4: {len(id_sets):,}")
total_ = len(df_rk_aiff_25_07)+ len(df_exports_25_7) +  len(df_STEMS_25_7) + len(df_samples_STEMS_25_7) 
print(len(df_rk_aiff_25_07),len(df_exports_25_7) , len(df_samples_STEMS_25_7),len(df_STEMS_25_7) )
total_

🎯 Total unique IDs across all 4: 77,950
1635 766 72263 3286


77950

# MERGE ALL IDS

In [41]:
# -----######-----###### MERGE ALL PATH + ID INTO ONE DF -----######-----###### #
import os
import pandas as pd

df_all_paths = pd.concat([
    df_rk_aiff_25_07[['ID', 'Path']],
    df_exports_25_7[['ID', 'Path']],
    df_STEMS_25_7[['ID', 'Path']],
    df_samples_STEMS_25_7[['id_stem_sample', 'Path']].rename(columns={'id_stem_sample': 'ID'})
], ignore_index=True)

# Check if Path exists
df_all_paths['path_exists'] = df_all_paths['Path'].apply(
    lambda x: os.path.exists(x) if pd.notna(x) else False
)

# Print summary
total = len(df_all_paths)
missing = (~df_all_paths['path_exists']).sum()

print(f"✅ Merged total rows     : {total:,}")
print(f"✅ Existing paths        : {total - missing:,}")
print(f"❌ Missing paths         : {missing:,}")
#print(df_all_paths.head(3))


✅ Merged total rows     : 77,950
✅ Existing paths        : 77,950
❌ Missing paths         : 0


In [42]:
df_all_paths.to_pickle('_df_77950_IDS_PATHS.pkl')

In [46]:
df_all_paths

,ID,Path,path_exists
0,t1-61300,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__...,True
1,t1-61301,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__...,True
2,t1-61302,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__...,True
3,t1-61303,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__...,True
4,t1-61304,/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__...,True
...,...,...,...
77945,s41_d6-7151c,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,True
77946,sb1_d6-7151c,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,True
77947,s43_d6-7151c,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,True
77948,s21_d6-7151c,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,True


# SHOW ALL COLUMNS 

In [28]:
# -----######-----###### DISPLAY COLUMNS OF ALL DFS -----######-----###### #
dfs = [
    ('df_rk_aiff_25_07', df_rk_aiff_25_07),
    ('df_exports_25_7', df_exports_25_7),
    ('df_STEMS_25_7', df_STEMS_25_7),
    ('df_samples_STEMS_25_7', df_samples_STEMS_25_7)
]

for name, df in dfs:
    print(f"\n🧾 {name} → {len(df.columns)} columns")
    print(df.columns.to_list())



🧾 df_rk_aiff_25_07 → 76 columns
['Path', 'temp_id', 'file_name', 'Extension', 'dur_seconds', 'dur_min', 'sr', 'bit_depth', 'bit_rate', 'channels', 'file_size', 'file_size_human', 'num_frames', 'error', 'ms_lufs', 'ms_LUFS_code', 'id_cat_lufs', 'mean_bpm', 'std_bpm', 'min_bpm', 'max_bpm', 'variation_percentage', 'dominant_bpm', 'bpm_consistency', 'bpm_consistency_cat', 'title', 'title_file', 'artist', 'artist_file', 'LABEL', 'label_file', 'genre', 'genre_file', 'rel_year', 'rel_year_file', 'KEY', 'key_file', 'mix_name', 'remixer', 'remix', 'date_purchased', 'rel_date', 'rel_date_file', 'key_dj', 'key_music', 'status', 'Relative_Key', 'Key_Up', 'Key_Down', 'Jaw_s_Mix', 'Mood_Shifter', 'ID', 'comment', 'Path_jpg_album', 'Path_jpg_key', 'Path_jpg_clip', 'Path_png_bar_dyn', 'ms_LUFS_norm', 'Path_png_bar_lufs', 'Spectral_Bandwidth', 'Spectral_Flatness', 'HEX_shape_texture', 'Path_png_bar_text', 'spec_centroid_hz', 'centroid_color', 'centroid_desc', 'Path_png_bar_centroid', 'Path_csv_freq', 

In [ ]:
# MAYBE ... 

In [30]:
#df_exports_25_7.to_pickle("_df_exports_766_25_07.pkl")

In [45]:
df_rk_aiff_25_07['ID']

0       t1-61300
1       t1-61301
2       t1-61302
3       t1-61303
4       t1-61304
          ...   
1630     t1-660k
1631     t1-660l
1632     t1-660m
1633     t1-660n
1634     t1-660o
Name: ID, Length: 1635, dtype: object